In [ ]:
import subprocess
# import boto3  # pipeline import
import os
import sys

In [ ]:
# BUCKET = "data-science-citation-network"  # pipeline bucket(?)

def parse_files_to_run():
    entries = []
    with open("FilesToRun.txt", "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.split("|")]
            if len(parts) == 2:
                notebook, s3_destination = parts
                entries.append((notebook, s3_destination))
            else:
                print(f"Skipping malformed line: {line}")
    return entries

In [ ]:
# pipeline code below:
# def upload_outputs(local_dir, s3_destination):
#     if not os.path.exists(local_dir):
#         print(f"  No output folder found at {local_dir}, skipping upload.")
#         return
#     s3 = boto3.client("s3")
#     for root, dirs, files in os.walk(local_dir):
#         for file in files:
#             local_path = os.path.join(root, file)
#             relative_path = os.path.relpath(local_path, local_dir)
#             s3_key = s3_destination + relative_path.replace("\\", "/")
#             print(f"  Uploading {local_path} -> s3://{BUCKET}/{s3_key}")
#             s3.upload_file(local_path, BUCKET, s3_key)

# Code to run files locally without S3 upload:
def upload_outputs(local_dir, s3_destination):
    print(f"  Skipping S3 upload — outputs saved locally in {local_dir}")

In [ ]:
def run_notebook(notebook, s3_destination):
    notebook_path = os.path.join("src", notebook)
    if not os.path.exists(notebook_path):
        print(f"ERROR: {notebook_path} not found. Skipping.")
        return False

    print(f"\n{'='*50}")
    print(f"Running: {notebook}")
    print(f"{'='*50}")

    result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace", notebook_path],
        capture_output=False
    )

    if result.returncode != 0:
        print(f"ERROR: {notebook} failed with exit code {result.returncode}")
        return False

    print(f"\nUploading results for {notebook}...")
    upload_outputs("outputs", s3_destination)
    return True

In [ ]:
entries = parse_files_to_run()
if not entries:
    print("FilesToRun.txt is empty or has no valid entries.")
else:
    print(f"Found {len(entries)} notebook(s) to run:")
    for notebook, dest in entries:
        print(f"  {notebook} -> {dest}")

    failed = []
    for notebook, s3_destination in entries:
        success = run_notebook(notebook, s3_destination)
        if not success:
            failed.append(notebook)

    if failed:
        print(f"\nThe following notebooks failed: {failed}")
    else:
        print("\nAll notebooks completed successfully!")